# Automated Test Suite for Deepfake Detection Pipeline\n

In [ ]:
"""
Automated Test Suite for Audio Deepfake Detection Pipeline
Tests:
1. Configuration loading and directory paths
2. Audio preprocessing transforms (resampling, mono, length equalization, normalization)
3. Dataset loading and tensor shapes
4. Model architecture forward pass
5. Exported checkpoint integrity and inference engine
"""

import unittest
from pathlib import Path
import numpy as np
import soundfile as sf
import torch

from src.config import config
from src.preprocess import process_single_audio
from src.dataset import DeepfakeAudioDataset
from src.model import AudioDeepfakeClassifier
from src.inference import AudioDeepfakeDetector

class TestDeepfakePipeline(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.test_dir = Path("tests/temp")
        cls.test_dir.mkdir(parents=True, exist_ok=True)
        # Create a dummy 44.1kHz stereo audio file (2.5 seconds)
        cls.dummy_sr = 44100
        cls.dummy_samples = int(cls.dummy_sr * 2.5)
        cls.dummy_data = np.random.uniform(-0.8, 0.8, (cls.dummy_samples, 2)).astype(np.float32)
        cls.dummy_audio_path = cls.test_dir / "test_stereo_input.wav"
        sf.write(str(cls.dummy_audio_path), cls.dummy_data, cls.dummy_sr)

    def test_01_config_integrity(self):
        """Tests that configuration contains valid paths and hyperparameters."""
        self.assertEqual(config.SAMPLE_RATE, 16000)
        self.assertEqual(config.TARGET_DURATION, 4.0)
        self.assertEqual(config.TARGET_SAMPLES, 64000)
        self.assertTrue(config.DATA_RAW_DIR.exists())
        self.assertTrue(config.MODELS_DIR.exists())

    def test_02_preprocessing_transforms(self):
        """Tests resampling to 16kHz, mono conversion, 4.0s equalization, and normalization."""
        out_path = self.test_dir / "test_preprocessed_output.wav"
        res = process_single_audio(
            self.dummy_audio_path,
            out_path,
            target_sr=config.SAMPLE_RATE,
            target_samples=config.TARGET_SAMPLES,
            norm_peak=config.NORM_MAX_PEAK
        )
        self.assertEqual(res["status"], "success")
        self.assertTrue(out_path.exists())

        # Verify output audio properties
        data, sr = sf.read(str(out_path))
        self.assertEqual(sr, 16000)
        self.assertEqual(data.ndim, 1)  # Mono
        self.assertEqual(len(data), 64000)  # Exactly 4.0s
        self.assertLessEqual(np.max(np.abs(data)), config.NORM_MAX_PEAK + 1e-4)

    def test_03_model_forward_pass(self):
        """Tests Wav2Vec2 classification model forward pass and output dimensions."""
        model = AudioDeepfakeClassifier(
            model_name_or_path=config.MODEL_NAME_OR_PATH,
            num_classes=2,
            hidden_dim=128
        ).to(config.DEVICE)
        model.eval()

        dummy_tensor = torch.randn(2, config.TARGET_SAMPLES).to(config.DEVICE)
        with torch.no_grad():
            out = model(dummy_tensor)

        self.assertIn("logits", out)
        self.assertIn("probs", out)
        self.assertEqual(out["logits"].shape, (2, 2))
        self.assertEqual(out["probs"].shape, (2, 2))
        # Probabilities should sum to 1.0 across classes
        prob_sums = out["probs"].sum(dim=-1).cpu().numpy()
        np.testing.assert_allclose(prob_sums, [1.0, 1.0], atol=1e-5)

    def test_04_exported_checkpoint_inference(self):
        """Tests that the exported .pth model can be loaded by AudioDeepfakeDetector and predict."""
        self.assertTrue(config.BEST_MODEL_PATH.exists(), "Trained model checkpoint not found!")
        detector = AudioDeepfakeDetector(model_path=config.BEST_MODEL_PATH)
        
        # Test on preprocessed sample
        data_clean_dir = config.DATA_PROCESSED_DIR
        sample_files = list(data_clean_dir.glob("**/*.wav"))
        self.assertGreater(len(sample_files), 0, "No preprocessed audio files found!")
        
        test_audio = sample_files[0]
        result = detector.predict(test_audio)
        self.assertIn(result["prediction"], ["Real", "Fake"])
        self.assertGreaterEqual(result["confidence"], 0.0)
        self.assertLessEqual(result["confidence"], 1.0)
        self.assertIn("Real", result["probabilities"])
        self.assertIn("Fake", result["probabilities"])

    @classmethod
    def tearDownClass(cls):
        import shutil
        if cls.test_dir.exists():
            shutil.rmtree(cls.test_dir)

if __name__ == "__main__":
    unittest.main()

\n